# 🚑 DeepShield — Demo Samples Rescue

The training session died before `demo_samples.zip` was saved. This mini-notebook regenerates it in ~10 minutes (same seed → same 24 images), and can optionally rebuild the confusion matrix for your trained model.

**Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**.

In [ ]:
# ── 1. Download the dataset (~8 min) ─────────────────────────────
import kagglehub, os
path = kagglehub.dataset_download('xhlulu/140k-real-and-fake-faces')
DATA = None
for root, dirs, _ in os.walk(path):
    if {'train', 'valid', 'test'} <= set(dirs):
        DATA = root
        break
assert DATA, 'dataset folders not found'
CLASSES = sorted(os.listdir(os.path.join(DATA, 'test')))  # ['fake', 'real']
print('Data root:', DATA, '| classes:', CLASSES)

In [ ]:
# ── 2. Rebuild demo_samples.zip (same seed = same 24 images) ─────
import shutil, random

DEMO = 'demo_samples'
shutil.rmtree(DEMO, ignore_errors=True)
os.makedirs(DEMO)

rng = random.Random(7)
picks = []
for cls in CLASSES:
    fnames = sorted(os.listdir(f'{DATA}/test/{cls}'))
    picks += [(cls, fn) for fn in rng.sample(fnames, 12)]
rng.shuffle(picks)

key_lines = []
for i, (cls, fn) in enumerate(picks, 1):
    out = f'sample_{i:02d}.jpg'
    shutil.copy(f'{DATA}/test/{cls}/{fn}', f'{DEMO}/{out}')
    key_lines.append(f'{out}  →  {cls.upper()}')

with open(f'{DEMO}/ANSWER_KEY.txt', 'w') as f:
    f.write('DeepShield demo samples — ground truth\n')
    f.write('(open only AFTER the demo!)\n\n')
    f.write('\n'.join(key_lines))

shutil.make_archive('demo_samples', 'zip', DEMO)
print(f'{len(picks)} blind samples + answer key ready')

from google.colab import files as colab_files
colab_files.download('demo_samples.zip')

---
## Optional — rebuild the confusion matrix for your V2 model

Run the next cell and, when the **Choose Files** button appears, upload your trained model from:
`G:\deepfake\models\deepshield_mobilenetv3.pth`

(Skip/stop here if you only wanted the demo samples.)

In [ ]:
# ── 3. (OPTIONAL) Upload model → test-set eval → confusion matrix ─
import torch
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from google.colab import files as colab_files

print('Upload deepshield_mobilenetv3.pth …')
up = colab_files.upload()
assert 'deepshield_mobilenetv3.pth' in up, 'model file not uploaded'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ck = torch.load('deepshield_mobilenetv3.pth', map_location=device, weights_only=False)
IMG  = ck.get('input_size', 224)
norm = ck.get('normalize', {'mean': [0.485, 0.456, 0.406], 'std': [0.229, 0.224, 0.225]})

model = models.mobilenet_v3_small(weights=None)
model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, len(ck['classes']))
model.load_state_dict(ck['state_dict'])
model = model.to(device).eval()
print('Model loaded — reported test accuracy:', ck.get('test_accuracy'), '%')

eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(norm['mean'], norm['std']),
])
test_dl = DataLoader(datasets.ImageFolder(f'{DATA}/test', eval_tf),
                     batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

from tqdm.auto import tqdm
import numpy as np
preds, trues = [], []
with torch.no_grad():
    for xb, yb in tqdm(test_dl, desc='Testing'):
        preds += model(xb.to(device)).argmax(1).cpu().tolist()
        trues += yb.tolist()
acc = (np.array(preds) == np.array(trues)).mean()
print(f'TEST accuracy (recomputed): {acc*100:.2f}%')

from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
print(classification_report(trues, preds, target_names=ck['classes']))
cm = confusion_matrix(trues, preds)
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black')
ax.set_xticks([0, 1], ck['classes']); ax.set_yticks([0, 1], ck['classes'])
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion matrix (V2 · {acc*100:.2f}%)')
plt.savefig('confusion_matrix.png', dpi=150); plt.show()
colab_files.download('confusion_matrix.png')

## ✅ Done
1. `demo_samples.zip` → extract into `g:\deepfake\training\demo_samples\`
2. `confusion_matrix.png` (if you ran the optional cell) → `g:\deepfake\training\results\`

Note: epoch-by-epoch training curves cannot be regenerated without retraining — for the report, use the accuracy numbers stored inside the checkpoint (val 99.40 · robust 98.54 · test 99.39).